<a href="https://colab.research.google.com/github/anupseth360-netizen/python-lab/blob/main/LAB_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IT12 — Rough Cut: Make the Log

In [ ]:

import os

os.makedirs("data", exist_ok=True)

records = [
    "PLANT1 | AB-1234 | 100",
    "PLANT1 | CD-2345 | 200",
    "PLANT2 | EF-3456 | 150",
    "PLANT2 | GH-4567 | 300",
    "PLANT1 | IJ-5678 | 250",
    "PLANT3 | KL-6789 | 180",
    "PLANT1 | MN-7890 | 120",
    "PLANT2 | OP-8901 | 220",
    "PLANT3 | QR-9012 | 160",
    "PLANT1 | ST-1234 | 190",
    "PLANT2 | UV-2345 | 140",
    "PLANT3 | WX-3456 | 210",

    # Three deliberately faulty records
    "PLANT1 | INVALID | 100",
    "PLANT2 | AB-4567 | -50",
    "PLANT3 | CD-7890",

    "PLANT1 | EF-1234 | 175",
    "PLANT2 | GH-5678 | 225"
]

file_path = "data/shift_log.txt"

with open(file_path, "w") as file:
    for record in records:
        file.write(record + "\n")

print("Shift log created successfully!")

with open(file_path, "r") as file:
    lines = file.readlines()

print("Total number of records:", len(lines))
print("\nFile contents:")

for line in lines:
    print(line.strip())


Shift log created successfully!
Total number of records: 17

File contents:
PLANT1 | AB-1234 | 100
PLANT1 | CD-2345 | 200
PLANT2 | EF-3456 | 150
PLANT2 | GH-4567 | 300
PLANT1 | IJ-5678 | 250
PLANT3 | KL-6789 | 180
PLANT1 | MN-7890 | 120
PLANT2 | OP-8901 | 220
PLANT3 | QR-9012 | 160
PLANT1 | ST-1234 | 190
PLANT2 | UV-2345 | 140
PLANT3 | WX-3456 | 210
PLANT1 | INVALID | 100
PLANT2 | AB-4567 | -50
PLANT3 | CD-7890
PLANT1 | EF-1234 | 175
PLANT2 | GH-5678 | 225


IT10 — Standard: The Full Pipeline

In [ ]:

import os
import re
import csv
from collections import defaultdict

os.makedirs("data", exist_ok=True)

input_file = "data/shift_log.txt"

report_file = "data/report.txt"
reject_file = "data/rejects.csv"

valid_records = []
rejected_records = []

# Read input file
with open(input_file, "r") as file:
    lines = file.readlines()

# Parse and validate records
for line_number, line in enumerate(lines, start=1):

    line = line.strip()

    if not line:
        continue

    parts = [p.strip() for p in line.split("|")]

    if len(parts) != 3:
        rejected_records.append(
            [line_number, line, "Incorrect number of fields"]
        )
        continue

    plant, part_code, quantity = parts

    if not re.fullmatch(r"[A-Z]{2}-\d{4}", part_code):
        rejected_records.append(
            [line_number, line, "Invalid part code"]
        )
        continue

    try:
        quantity = int(quantity)
    except ValueError:
        rejected_records.append(
            [line_number, line, "Quantity must be an integer"]
        )
        continue

    if quantity <= 0:
        rejected_records.append(
            [line_number, line, "Quantity must be positive"]
        )
        continue

    valid_records.append((plant, part_code, quantity))

# Aggregate records
summary = defaultdict(int)

for plant, part_code, quantity in valid_records:
    summary[plant] += quantity

# Generate human-readable report
with open(report_file, "w") as file:

    file.write("SHIFT PRODUCTION REPORT\n")
    file.write("=" * 45 + "\n")

    file.write(f"Total input lines: {len(lines)}\n")
    file.write(f"Valid records: {len(valid_records)}\n")
    file.write(f"Rejected records: {len(rejected_records)}\n\n")

    file.write("PLANT          TOTAL QUANTITY\n")
    file.write("-" * 35 + "\n")

    for plant, total in sorted(summary.items()):
        file.write(f"{plant:<15}{total}\n")

# Generate rejected records CSV
with open(reject_file, "w", newline="") as file:

    writer = csv.writer(file)

    writer.writerow([
        "Line Number",
        "Original Record",
        "Reason"
    ])

    writer.writerows(rejected_records)

print("Pipeline completed successfully!")
print("Valid records:", len(valid_records))
print("Rejected records:", len(rejected_records))

print("\nOutput files:")
print(report_file)
print(reject_file)


Pipeline completed successfully!
Valid records: 14
Rejected records: 3

Output files:
data/report.txt
data/rejects.csv


IT8 — Fine: Regex on Real Text

In [ ]:
# New regex-based parsing method
def regex_parser(line):

    pattern = (
        r"^\s*"
        r"([A-Z0-9]+)\s*\|\s*"
        r"([A-Z]{2}-\d{4})\s*\|\s*"
        r"(\d+)"
        r"\s*$"
    )

    match = re.fullmatch(pattern, line)

    if match:
        plant, part_code, quantity = match.groups()

        quantity = int(quantity)

        if quantity <= 0:
            return None

        return plant, part_code, quantity

    return None


# Test records
test_records = [
    "PLANT1 | AB-1234 | 100",
    "PLANT2|CD-2345|200",
    "PLANT3  |  EF-3456  |  150",
    "PLANT1 | INVALID | 100",
    "PLANT2 | GH-4567 | -50",
    "PLANT3 | IJ-1234"
]

IT6 — Precision: The Batch Job

In [ ]:

import os
import re
import csv
from datetime import datetime
from collections import defaultdict

data_folder = "data"

os.makedirs(data_folder, exist_ok=True)

combined_summary = defaultdict(int)

total_processed = 0
total_rejected = 0

history_file = os.path.join(
    data_folder, "run_history.csv"
)

# Process every text file in data folder
for filename in os.listdir(data_folder):

    if not filename.endswith(".txt"):
        continue

    input_path = os.path.join(data_folder, filename)

    valid_records = []
    rejected_records = []

    with open(input_path, "r") as file:
        lines = file.readlines()

    for line_number, line in enumerate(lines, start=1):

        line = line.strip()

        if not line:
            continue

        parts = [p.strip() for p in line.split("|")]

        if len(parts) != 3:
            rejected_records.append(line)
            continue

        plant, part_code, quantity = parts

        if not re.fullmatch(r"[A-Z]{2}-\d{4}", part_code):
            rejected_records.append(line)
            continue

        try:
            quantity = int(quantity)
        except ValueError:
            rejected_records.append(line)
            continue

        if quantity <= 0:
            rejected_records.append(line)
            continue

        valid_records.append((plant, part_code, quantity))

        combined_summary[plant] += quantity

    processed = len(valid_records)
    rejected = len(rejected_records)

    total_processed += processed
    total_rejected += rejected

    # Create individual report
    report_name = filename.replace(".txt", "_report.txt")

    report_path = os.path.join(data_folder, report_name)

    with open(report_path, "w") as report:

        report.write("FILE PROCESSING REPORT\n")
        report.write("=" * 35 + "\n")

        report.write(f"Input file: {filename}\n")
        report.write(f"Processed records: {processed}\n")
        report.write(f"Rejected records: {rejected}\n\n")

        for plant, part_code, quantity in valid_records:
            report.write(
                f"{plant} | {part_code} | {quantity}\n"
            )

    print("Processed:", filename)
    print("Valid:", processed)
    print("Rejected:", rejected)


# Create combined plant summary
summary_path = os.path.join(
    data_folder, "combined_summary.txt"
)

with open(summary_path, "w") as file:

    file.write("COMBINED PLANT SUMMARY\n")
    file.write("=" * 35 + "\n")

    for plant, total in sorted(combined_summary.items()):
        file.write(f"{plant:<15} {total}\n")

# Append execution history
file_exists = os.path.exists(history_file)

with open(history_file, "a", newline="") as file:

    writer = csv.writer(file)

    if not file_exists:
        writer.writerow([
            "Timestamp",
            "Filename",
            "Records Processed",
            "Rejected Records"
        ])

    timestamp = datetime.now().isoformat(
        timespec="seconds"
    )

    # One history entry for this batch run
    writer.writerow([
        timestamp,
        "ALL_TXT_FILES",
        total_processed,
        total_rejected
    ])

print("\nBATCH JOB COMPLETED")
print("=" * 35)
print("Total processed:", total_processed)
print("Total rejected:", total_rejected)
print("Combined summary:", summary_path)
print("Run history:", history_file)


Processed: shift_log.txt
Valid: 14
Rejected: 3
Processed: report.txt
Valid: 0
Rejected: 10

BATCH JOB COMPLETED
Total processed: 14
Total rejected: 13
Combined summary: data/combined_summary.txt
Run history: data/run_history.csv
